# Elicitation, Entropy and Binding

Pythia: 160M, 410M, 1B, 2.8B, 12B   
GPT-2: Small, Medium, Large, XL  
OLMo 2: 1B, 7B, 13B

## Setup

In [ ]:
# Colab Stuff
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [ ]:
# Imports
import sys
import torch
from pathlib import Path

# Colab: files are in /content/
# Local: notebook is in notebooks/, project root is one level up
if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

In [ ]:
# Model name variable
model_name = "EleutherAI/pythia-160m"

In [ ]:
# Cell 3: Load Model
model = HookedTransformer.from_pretrained(model_name)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

### Experiment 1: Prompts

In [ ]:
# Experiment 1 — Run all prompts
import importlib
import src.elicitation
importlib.reload(src.elicitation)
from src.elicitation import run_all_prompts
results_df = run_all_prompts(model, model_name, PROJECT_ROOT)
results_df

In [ ]:
# Experiment 2 — Entropy
from src.entropy import run_entropy_analysis
entropy_df = run_entropy_analysis(model, model_name, PROJECT_ROOT)

In [ ]:
# Experiment 3 — Binding
from src.binding import run_binding_sweep
binding_df = run_binding_sweep(model, model_name, PROJECT_ROOT)

### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")